# 04 - Objective evaluation path on a mock objective

This notebook confirms the structure of a tuner's objective by subclassing the
real `Phase1Tuner` (from `pipelines.tuning_pipeline.tuners`) and replacing only
the expensive training call with a cheap synthetic objective whose optimum is
known. The full path is exercised: a fresh model config is built,
`_apply_params` samples and applies the parameters, and the synthetic loss is
returned to Optuna.

The synthetic loss is a smooth function of `encoder_lr` and `dropout` with a
known minimum, so we can verify the tuner drives the sampled parameters toward
that optimum.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt
import optuna

import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams.update({
    "figure.figsize" : (7.0, 4.2),
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
})


## A Phase1Tuner with a synthetic objective

We override `_objective` to reuse the genuine config construction and parameter
application, then evaluate a closed-form loss instead of training. The optimum
sits at `encoder_lr = 1e-3` (log distance) and `dropout = 0.2`.

In [ ]:
from pipelines.tuning_pipeline.tuners import Phase1Tuner
from configuration.models_config    import UNetConfig
from configuration.tuning_config     import Phase1TuneConfig

LR_STAR      = 1e-3
DROPOUT_STAR = 0.2

class NullLogger:
    def __getattr__(self, _):
        return lambda *a, **k: None

class MockPhase1Tuner(Phase1Tuner):
    def _objective(self, trial):
        model_config = self.model_config_cls()
        self._apply_params(trial, model_config)

        lr_term  = (np.log10(model_config.encoder_lr) - np.log10(LR_STAR)) ** 2
        do_term  = (model_config.dropout - DROPOUT_STAR) ** 2
        loss     = float(lr_term + 8.0 * do_term)

        trial.set_user_attr('encoder_lr', model_config.encoder_lr)
        trial.set_user_attr('dropout',    model_config.dropout)
        return loss

tuner = MockPhase1Tuner(
    model_name          = 'unet',
    model_config_cls    = UNetConfig,
    base_trainer_config = None,
    base_dataset_config = None,
    tune_cfg            = Phase1TuneConfig(),
    log_dir             = '/tmp/nb_tuning_unused',
    logger              = NullLogger(),
)
print('mock tuner ready')

## Run the study through the real .run entry point

We create a study exactly as the orchestrator does (TPE sampler, minimize) and
call `tuner.run`, which internally calls `study.optimize(self._objective, ...)`.
This confirms the objective is wired through the genuine `BaseTuner.run`.

In [ ]:
study = optuna.create_study(
    direction = 'minimize',
    sampler   = optuna.samplers.TPESampler(seed=42, n_startup_trials=10),
)

tuner.run(study, n_trials=120)

best = study.best_trial
print('best value     :', round(best.value, 5))
print('best encoder_lr:', best.user_attrs['encoder_lr'])
print('best dropout   :', round(best.user_attrs['dropout'], 4))
print('targets        : encoder_lr=%.0e dropout=%.2f' % (LR_STAR, DROPOUT_STAR))

## Sampled points relative to the optimum

Each completed trial is plotted in the (log encoder_lr, dropout) plane, coloured
by loss. The known optimum is marked. Lower-loss points should cluster around
it, confirming the search concentrates on the right region.

In [ ]:
lrs    = np.array([t.user_attrs['encoder_lr'] for t in study.trials])
drops  = np.array([t.user_attrs['dropout']    for t in study.trials])
losses = np.array([t.value for t in study.trials])

fig, ax = plt.subplots()
sc = ax.scatter(np.log10(lrs), drops, c=losses, cmap='viridis_r', s=28, edgecolor='k', linewidth=0.3)
ax.scatter([np.log10(LR_STAR)], [DROPOUT_STAR], marker='*', s=320, color='red', edgecolor='k', label='optimum', zorder=5)
ax.set_xlabel('log10(encoder_lr)')
ax.set_ylabel('dropout')
ax.set_title('Sampled points coloured by synthetic loss')
ax.legend(loc='upper right')
fig.colorbar(sc, ax=ax).set_label('loss')
fig.tight_layout()
plt.show()

## Expected visual outcome

The best trial lands close to `encoder_lr = 1e-3` and `dropout = 0.2`. The
scatter shows darker (low-loss) points clustering around the red star, while
the early random startup trials are scattered more widely, confirming the
objective path drives the search toward the known optimum.